# SOC与能耗关系的连续时间耦合模型
## Continuous-Time SOC-Energy Coupled Model

本笔记本演示如何使用马尔科夫时间分区和微分耦合方程组进行:
1. SOC与能耗关系的连续时间建模
2. 基于用户使用情况预测电池剩余使用时间

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from soc_energy_coupled_model import (
    SOCEnergyCoupledSystem,
    BatteryLifePredictor,
    BatteryParams,
    HardwareParams,
    visualize_simulation_results,
    visualize_prediction
)

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("模型加载完成!")

## 1. 模型架构说明

### 1.1 核心微分方程组

电池电化学-热耦合方程:

$$\frac{dSOC(t)}{dt} = -\frac{I_{total}(t)}{Q_{max} \cdot N}$$

$$\frac{dT_{batt}}{dt} = \frac{P_{joule} + P_{entropy} - (T_{batt} - T_{env})/R_{th}}{C_{th}}$$

其中:
- $V_{batt} = V_{OCV}(SOC) - I_{total} \cdot R_{int}(SOC, T)$
- $P_{joule} = I_{total}^2 \cdot R_{int}$
- $P_{entropy} = I_{total} \cdot T \cdot \frac{dV_{OCV}}{dT}$

### 1.2 连续时间马尔科夫用户行为模型

Kolmogorov前向方程:

$$\frac{dp(t)}{dt} = p(t) \cdot Q(t)$$

其中Q(t)为时变转移速率矩阵，根据时段(睡眠/工作/休闲)动态切换。

## 2. 创建模型实例

In [ ]:
# 自定义电池参数
battery_params = BatteryParams(
    Q_max=4000.0,        # 4000mAh电池容量
    V_nominal=3.7,       # 标称电压
    R_int_25=0.08,       # 25°C时内阻
    C_th=50.0,           # 热容
    R_th=10.0,           # 热阻
    T_env=25.0           # 环境温度
)

# 创建系统和预测器
system = SOCEnergyCoupledSystem(battery_params=battery_params)
predictor = BatteryLifePredictor(system)

print("电池参数:")
print(f"  容量: {battery_params.Q_max} mAh")
print(f"  标称电压: {battery_params.V_nominal} V")
print(f"  额定能量: {battery_params.Q_max * battery_params.V_nominal / 1000:.1f} Wh")

## 3. 用户行为马尔科夫模型分析

In [ ]:
# 分析不同时段的用户状态分布演化
user_model = system.user_model

# 初始状态分布 (从轻度使用开始)
initial_dist = np.array([0.2, 0.5, 0.2, 0.1])

# 模拟24小时的状态演化
time_axis, prob_history = user_model.simulate_state_distribution(
    initial_dist, duration_hours=24, start_hour=0
)

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 状态概率分布
ax1 = axes[0]
states = ['Deep Sleep', 'Light Use', 'Streaming', 'Gaming']
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

ax1.stackplot(time_axis, prob_history.T, labels=states, colors=colors, alpha=0.8)
ax1.set_ylabel('State Probability', fontsize=11)
ax1.set_xlim([0, 24])
ax1.set_ylim([0, 1])
ax1.legend(loc='upper right', ncol=2)
ax1.set_title('User State Distribution Over 24 Hours (Markov Chain)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 添加时段标记
time_periods = [
    (0, 7, 'Sleep', 'lightgray'),
    (7, 9, 'Leisure', 'lightgreen'),
    (9, 12, 'Work', 'lightyellow'),
    (12, 14, 'Leisure', 'lightgreen'),
    (14, 18, 'Work', 'lightyellow'),
    (18, 23, 'Leisure', 'lightgreen'),
    (23, 24, 'Sleep', 'lightgray')
]

ax2 = axes[1]
for start, end, label, color in time_periods:
    ax2.axvspan(start, end, alpha=0.3, color=color)
    ax2.text((start + end) / 2, 0.5, label, ha='center', fontsize=10)

# 绘制各状态概率曲线
for i, (state, color) in enumerate(zip(states, colors)):
    ax2.plot(time_axis, prob_history[:, i], color=color, linewidth=2, label=state)

ax2.set_xlabel('Time of Day (hours)', fontsize=11)
ax2.set_ylabel('Probability', fontsize=11)
ax2.set_xlim([0, 24])
ax2.set_ylim([0, 1])
ax2.legend(loc='upper right')
ax2.set_title('Individual State Probabilities', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 25, 2))

plt.tight_layout()
plt.show()

## 4. SOC与能耗关系仿真

In [ ]:
# 仿真24小时电池放电
print("开始24小时电池放电仿真...")

results = system.simulate_battery_drain(
    initial_SOC=1.0,
    initial_T=25.0,
    duration_hours=24,
    start_hour=8  # 早上8点开始
)

print(f"\n仿真结果:")
print(f"  初始SOC: 100%")
print(f"  最终SOC: {results['SOC'][-1]*100:.1f}%")
print(f"  实际仿真时长: {results['time'][-1]:.1f} 小时")
print(f"  平均功耗: {np.mean(results['power']):.2f} W")
print(f"  峰值功耗: {np.max(results['power']):.2f} W")
print(f"  最低功耗: {np.min(results['power']):.2f} W")
print(f"  温度范围: {results['temperature'].min():.1f}°C - {results['temperature'].max():.1f}°C")

In [ ]:
# 详细可视化
fig = visualize_simulation_results(results)
plt.show()

## 5. SOC-能耗关系分析

In [ ]:
# 分析SOC与功耗的关系
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. SOC vs Power
ax1 = axes[0, 0]
scatter = ax1.scatter(results['SOC'] * 100, results['power'], 
                      c=results['time'], cmap='viridis', s=10, alpha=0.6)
ax1.set_xlabel('SOC (%)')
ax1.set_ylabel('Power (W)')
ax1.set_title('SOC vs Power Consumption')
plt.colorbar(scatter, ax=ax1, label='Time (hours)')
ax1.grid(True, alpha=0.3)

# 2. 累积能量消耗
ax2 = axes[0, 1]
# 计算累积能量 (Wh)
dt = np.diff(results['time'], prepend=0)  # 时间间隔
energy_consumed = np.cumsum(results['power'] * dt)  # Wh

ax2.plot(results['time'], energy_consumed, 'b-', linewidth=2)
ax2.set_xlabel('Time (hours)')
ax2.set_ylabel('Cumulative Energy (Wh)')
ax2.set_title('Cumulative Energy Consumption')
ax2.grid(True, alpha=0.3)

# 3. SOC变化率
ax3 = axes[1, 0]
dSOC_dt = np.gradient(results['SOC'], results['time']) * 100  # %/hour
ax3.plot(results['time'], -dSOC_dt, 'r-', linewidth=1.5)
ax3.set_xlabel('Time (hours)')
ax3.set_ylabel('SOC Drain Rate (%/hour)')
ax3.set_title('SOC Drain Rate Over Time')
ax3.grid(True, alpha=0.3)

# 4. 电压与SOC关系
ax4 = axes[1, 1]
ax4.plot(results['SOC'] * 100, results['voltage'], 'g-', linewidth=2)
ax4.set_xlabel('SOC (%)')
ax4.set_ylabel('Battery Voltage (V)')
ax4.set_title('Battery Voltage vs SOC')
ax4.grid(True, alpha=0.3)
ax4.set_xlim([0, 100])

plt.tight_layout()
plt.show()

print(f"总能量消耗: {energy_consumed[-1]:.2f} Wh")
print(f"平均SOC消耗率: {(1 - results['SOC'][-1]) * 100 / results['time'][-1]:.1f} %/hour")

## 6. 电池剩余时间预测

In [ ]:
# 预测电池剩余使用时间
print("=" * 50)
print("电池剩余使用时间预测")
print("=" * 50)

# 测试不同SOC水平的预测
test_cases = [
    {'SOC': 0.80, 'T': 25.0, 'hour': 9, 'desc': '早上9点，SOC 80%'},
    {'SOC': 0.50, 'T': 28.0, 'hour': 14, 'desc': '下午2点，SOC 50%'},
    {'SOC': 0.30, 'T': 30.0, 'hour': 20, 'desc': '晚上8点，SOC 30%'},
    {'SOC': 0.15, 'T': 32.0, 'hour': 22, 'desc': '晚上10点，SOC 15%'},
]

for case in test_cases:
    prediction = predictor.predict_remaining_time(
        current_SOC=case['SOC'],
        current_T=case['T'],
        current_hour=case['hour']
    )
    
    print(f"\n{case['desc']}:")
    print(f"  预测剩余时间: {prediction['remaining_hours']:.1f} 小时 ({prediction['remaining_minutes']:.0f} 分钟)")
    print(f"  平均功耗: {prediction['average_power_W']:.2f} W")
    print(f"  预测置信度: {prediction['confidence']*100:.1f}%")

In [ ]:
# 可视化一个详细预测
prediction = predictor.predict_remaining_time(
    current_SOC=0.50,
    current_T=28.0,
    current_hour=14
)

fig = visualize_prediction(prediction)
plt.show()

## 7. 多场景预测对比

In [ ]:
# 多场景预测 (乐观/中性/悲观)
print("=" * 50)
print("多场景预测对比 (SOC 60%, 10:00AM)")
print("=" * 50)

scenarios = predictor.predict_with_scenarios(
    current_SOC=0.60,
    current_T=28.0,
    current_hour=10
)

# 创建对比图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scenario_names = ['optimistic', 'neutral', 'pessimistic']
scenario_labels = ['乐观 (轻度使用)', '中性 (混合使用)', '悲观 (重度使用)']
colors_scenario = ['green', 'blue', 'red']

for idx, (name, label, color) in enumerate(zip(scenario_names, scenario_labels, colors_scenario)):
    pred = scenarios[name]
    results_s = pred['simulation_results']
    
    ax = axes[idx]
    ax.plot(results_s['time'], results_s['SOC'] * 100, color=color, linewidth=2)
    ax.axhline(y=5, color='red', linestyle='--', alpha=0.5)
    ax.axvline(x=pred['remaining_hours'], color='gray', linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('SOC (%)')
    ax.set_title(f'{label}\n剩余: {pred["remaining_hours"]:.1f}h, 功耗: {pred["average_power_W"]:.2f}W')
    ax.set_ylim([0, 70])
    ax.grid(True, alpha=0.3)
    
    print(f"\n{label}:")
    print(f"  剩余时间: {pred['remaining_hours']:.1f} 小时")
    print(f"  平均功耗: {pred['average_power_W']:.2f} W")
    print(f"  状态分布: {np.round(pred['state_distribution'] * 100, 1)}%")

plt.tight_layout()
plt.show()

## 8. 用户使用模式对电池寿命的影响分析

In [ ]:
# 分析不同使用模式对电池寿命的影响
print("=" * 50)
print("用户使用模式对电池寿命的影响")
print("=" * 50)

# 定义不同用户画像
user_profiles = {
    '轻度用户': np.array([0.5, 0.35, 0.1, 0.05]),   # 主要睡眠和轻度使用
    '普通用户': np.array([0.25, 0.35, 0.25, 0.15]), # 均衡使用
    '媒体爱好者': np.array([0.15, 0.2, 0.5, 0.15]), # 大量流媒体
    '游戏玩家': np.array([0.1, 0.15, 0.25, 0.5]),   # 重度游戏
}

fig, ax = plt.subplots(figsize=(12, 6))

profile_results = {}
colors_profile = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

for idx, (name, state_dist) in enumerate(user_profiles.items()):
    result = system.simulate_battery_drain(
        initial_SOC=1.0,
        initial_T=25.0,
        duration_hours=48,
        start_hour=8,
        initial_state_dist=state_dist
    )
    
    profile_results[name] = result
    ax.plot(result['time'], result['SOC'] * 100, 
            color=colors_profile[idx], linewidth=2, label=name)
    
    # 计算电池耗尽时间
    below_5 = np.where(result['SOC'] <= 0.05)[0]
    if len(below_5) > 0:
        drain_time = result['time'][below_5[0]]
    else:
        drain_time = result['time'][-1]
    
    print(f"{name}: 电池耗尽时间约 {drain_time:.1f} 小时, 平均功耗 {np.mean(result['power']):.2f} W")

ax.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='关机阈值 (5%)')
ax.set_xlabel('Time (hours)', fontsize=11)
ax.set_ylabel('SOC (%)', fontsize=11)
ax.set_title('不同用户使用模式的电池SOC变化', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])
ax.set_ylim([0, 105])

plt.tight_layout()
plt.show()

## 9. 模型参数敏感性分析

In [ ]:
# 分析电池容量和温度对剩余时间的影响
print("=" * 50)
print("模型参数敏感性分析")
print("=" * 50)

# 1. 电池容量影响
capacities = [3000, 4000, 5000, 6000]  # mAh

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
for cap in capacities:
    params = BatteryParams(Q_max=cap)
    sys = SOCEnergyCoupledSystem(battery_params=params)
    result = sys.simulate_battery_drain(
        initial_SOC=1.0, duration_hours=48, start_hour=8
    )
    ax1.plot(result['time'], result['SOC'] * 100, 
             linewidth=2, label=f'{cap} mAh')

ax1.axhline(y=5, color='red', linestyle='--', alpha=0.5)
ax1.set_xlabel('Time (hours)')
ax1.set_ylabel('SOC (%)')
ax1.set_title('电池容量对SOC变化的影响')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 30])

# 2. 环境温度影响
temps = [15, 25, 35, 45]  # °C

ax2 = axes[1]
for temp in temps:
    params = BatteryParams(T_env=temp)
    sys = SOCEnergyCoupledSystem(battery_params=params)
    result = sys.simulate_battery_drain(
        initial_SOC=1.0, initial_T=temp, duration_hours=24, start_hour=8
    )
    ax2.plot(result['time'], result['SOC'] * 100, 
             linewidth=2, label=f'{temp}°C')

ax2.axhline(y=5, color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel('Time (hours)')
ax2.set_ylabel('SOC (%)')
ax2.set_title('环境温度对SOC变化的影响')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. 总结

### 模型特点:
1. **连续时间马尔科夫链**: 建模用户行为的时间演化，考虑不同时段(睡眠/工作/休闲)的状态转移特性
2. **电化学-热耦合方程组**: 描述SOC、温度、电压之间的耦合关系
3. **模块化功耗计算**: 分别计算SoC、显示、通信等模块的功耗
4. **电池剩余时间预测**: 基于当前状态和用户使用模式预测电池可用时间

### 核心公式:
- SOC微分方程: $\frac{dSOC}{dt} = -\frac{I_{total}}{Q_{max}}$
- 总功耗: $P_{total} = P_{SoC} + P_{display} + P_{5G} + P_{BT} + P_{GNSS} + P_{background}$
- Kolmogorov方程: $\frac{dp}{dt} = p \cdot Q(t)$